# Q6. Jitter-corrected spike cross-correlograms

Run `q6_ccg_jittered.py` and `q6_ccg_jittered_stats.R` before running this notebook. It saves figures in `results/q6/ccg_jittered`. Set `Q6_CCG_OUTPUT_DIR` to use a different run.

Positive lag means the target follows the source. The 5 ms lag bins use population spike counts from the same matched-unit draws as the other Q6 analyses. For each pair, we subtract the exact expected cross-count under independent uniform jitter, preserving spike counts in each 200 ms interval, and divide by the product of the population count series' centered Euclidean norms. The direction score averages the corrected CCG at +5 to +50 ms and subtracts its average at the mirrored negative lags. Zero lag is excluded. Bands show pointwise 95% intervals across mice. [Interval-jitter method](https://pmc.ncbi.nlm.nih.gov/articles/PMC3349623/)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t

if 'Q6_CCG_OUTPUT_DIR' in os.environ:
    output = Path(os.environ['Q6_CCG_OUTPUT_DIR'])
else:
    repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / 'code/questions/q6_ccg_jittered.py').is_file())
    output = repo / 'results/q6/ccg_jittered'
curves = pd.read_csv(output / 'q6_ccg_mouse_correlograms.csv')
means = pd.read_csv(output / 'q6_ccg_stats_means.csv')
tests = pd.read_csv(output / 'q6_ccg_stats_primary_tests.csv')
curves['direction'] = curves['source'].astype(str) + '->' + curves['target'].astype(str)
colors = {'1->2': '#1B6CA8', '3->2': '#A24A12', '3->1': '#6B4B91'}

def label(direction):
    return direction.replace('->', ' → ')

display(means.loc[means['primary'] & means['candidate']])
display(tests.loc[tests['adjustment'] == 'Tukey'])

In [ ]:
primary_curves = curves.query('jitter_ms == 200 and candidate')
summary = primary_curves.groupby(['direction', 'lag_ms']).corrected_ccg.agg(['mean', 'std', 'count'])
fig, axes = plt.subplots(1, 3, figsize=(10, 3.7), sharex=True, sharey=True, layout='constrained')
for ax, (direction, color) in zip(axes, colors.items()):
    data = summary.loc[direction]
    ci = t.ppf(0.975, data['count'] - 1) * data['std'] / np.sqrt(data['count'])
    ax.axvspan(-52.5, -2.5, color='0.92', zorder=0)
    ax.axvspan(2.5, 52.5, color='0.92', zorder=0)
    ax.axvline(0, ls=':', color='0.55', lw=1)
    ax.axhline(0, ls='--', color='0.55', lw=1)
    ax.fill_between(data.index, data['mean'] - ci, data['mean'] + ci, color=color, alpha=0.2)
    ax.plot(data.index, data['mean'], color=color, lw=1.7)
    ax.set(title=label(direction), xlabel='Lag (ms)')
axes[0].set_ylabel('Normalized excess co-spiking')
fig.suptitle('Jitter-corrected cross-correlograms · 200 ms jitter')
fig.savefig(output / 'q6_ccg_correlograms.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
candidates = means.loc[means['candidate']]
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=True, layout='constrained')
panels = [(candidates.query('max_lag_ms == 50'), 'jitter_ms', 200, 'Jitter window (ms)', 'Lags through 50 ms'),
          (candidates.query('jitter_ms == 200'), 'max_lag_ms', 50, 'Maximum mirrored lag (ms)', '200 ms jitter')]
for ax, (data, key, primary_value, xlabel, title) in zip(axes, panels):
    settings = sorted(data[key].unique())
    primary_idx = settings.index(primary_value)
    ax.axvspan(primary_idx - 0.23, primary_idx + 0.23, color='0.93', zorder=0)
    ax.axhline(0, ls='--', color='0.55', lw=1)
    for i, (direction, color) in enumerate(colors.items()):
        rows = data.loc[data['direction'] == direction].sort_values(key)
        x = np.array([settings.index(value) for value in rows[key]]) + (i - 1) * 0.14
        ax.errorbar(x, rows.estimate, yerr=[rows.estimate - rows.ci95_low, rows.ci95_high - rows.estimate],
                    fmt='o-', capsize=3, color=color, label=label(direction))
    ax.set_xticks(range(len(settings)), settings)
    ax.set(xlabel=xlabel, title=title, xlim=(-0.33, len(settings) - 0.67))
axes[0].set_ylabel('Mean positive − negative lag score')
axes[1].legend(frameon=False, title='Source → target', loc='upper left', bbox_to_anchor=(1.01, 1))
fig.suptitle('Sensitivity of lead–lag asymmetry')
fig.savefig(output / 'q6_ccg_sensitivity.png', dpi=180, bbox_inches='tight')
plt.show()